In [2]:
from pyspark.sql.functions import (col, when, trim, upper, lower, to_date,
    month, year, quarter, dayofweek,
    round as spark_round, lit, current_timestamp)


# read the delta table 

df_silver = spark.read.format("delta").table("bronze_transactions")

print(f"Starting row count: {df_silver.count()}")

StatementMeta(, 1b496063-d1ae-4451-bb24-ea4e5ceacd3d, 4, Finished, Available, Finished, False)

Starting row count: 10100


In [3]:
# drop duplicates

df_silver = df_silver.dropDuplicates(["transaction_id"])

print(f"After dedup: {df_silver.count()}")

StatementMeta(, 1b496063-d1ae-4451-bb24-ea4e5ceacd3d, 5, Finished, Available, Finished, False)

After dedup: 10000


In [6]:
# Handle Nulls


# drop rows with null amount , they are unusable
df_silver = df_silver.filter(col("amount").isNotNull())

# Label Null category as Uncategorized
df_silver = df_silver.fillna({"category" : "Uncategorized"})

# drop rows where status is fail

df_silver = df_silver.filter(col("status") != "failed")

print(f"After null handling & status filter: {df_silver.count()} rows")
display(df_silver)

StatementMeta(, 1b496063-d1ae-4451-bb24-ea4e5ceacd3d, 8, Finished, Available, Finished, False)

After null handling & status filter: 7808 rows


SynapseWidget(Synapse.DataFrame, 9bf4f95e-d494-4e06-b3a6-2be07a40da92)

In [7]:
# Standardize and clean columns

df_silver = df_silver \
    .withColumn("merchant", trim(col("merchant"))) \
    .withColumn("category", trim(col("category"))) \
    .withColumn("status", lower(trim(col("status")))) \
    .withColumn("currency", upper(trim(col("currency")))) \
    .withColumn("amount", spark_round(col("amount"), 2)) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

StatementMeta(, 1b496063-d1ae-4451-bb24-ea4e5ceacd3d, 9, Finished, Available, Finished, False)

In [12]:
df_silver = df_silver \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("quarter", quarter(col("date"))) \
    .withColumn("day_of_week", dayofweek(col("date"))) \
    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), True).otherwise(False)) \
    .withColumn("amount_bucket",
        when(col("amount") < 20, "Small")
        .when(col("amount") < 100, "Medium")
        .when(col("amount") < 250, "Large")
        .otherwise("Very Large")
    ) \
    .withColumn("silver_timestamp", current_timestamp()) \
    .withColumn("layer", lit("silver"))

df_silver.printSchema()

StatementMeta(, 1b496063-d1ae-4451-bb24-ea4e5ceacd3d, 14, Finished, Available, Finished, False)

root
 |-- transaction_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = false)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- layer: string (nullable = false)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: boolean (nullable = false)
 |-- amount_bucket: string (nullable = false)
 |-- silver_timestamp: timestamp (nullable = false)



In [13]:
# Drop columns not usefull moving forwrd

columns_to_drop = ["sourcefile", "ingestion_timestamp"]
df_silver = df_silver.drop(*columns_to_drop)

print(f"Final Silver columns: {df_silver.columns}")

StatementMeta(, 1b496063-d1ae-4451-bb24-ea4e5ceacd3d, 15, Finished, Available, Finished, False)

Final Silver columns: ['transaction_id', 'date', 'merchant', 'category', 'amount', 'currency', 'status', 'layer', 'year', 'month', 'quarter', 'day_of_week', 'is_weekend', 'amount_bucket', 'silver_timestamp']


In [15]:
# write to delta table

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_transactions")
print("Silver Delta table created successfully")    

StatementMeta(, 1b496063-d1ae-4451-bb24-ea4e5ceacd3d, 17, Finished, Available, Finished, False)

Silver Delta table created successfully
